# Bronze - VRA (Voo Regular Ativo)
### Lê os 12 CSVs mensais do volume voebem.bronze.arquivos/vra/ e materializa voebem.bronze.vra

Regras da camada Bronze:
- Nada de tipagem - tudo string, exatamente como veio do arquivo;
- Nada de filtro - nenhuma linha é descartada;
- Colunas de auditoria - de cada arquivo veio e quando foi ingerido;
- idempotente - rodar duas vezes não duplica.

In [0]:
from pyspark.sql import functions as F

CAMINHO = "/Volumes/voebem/bronze/arquivos/vra/*.csv"
TABELA = "voebem.bronze.vra"





# Leitura

Quatro opções resolvem problemas de arquivo:

| opção | resolve |
|-------|----------|
| sep=";" | separador brasileiro, não vírgula |
| skipRows=1 | a 1ª linha é Atualizado em: \<data\>, não o cabeçalho — e o BOM tá aa aí também, some junto |
| header=true | a 2ª linha (a primeira que sobra) é o cabeçalho de verdade |
| inferSchema **desligado** (default) | bronze não tipa: tudo chega como string |




In [0]:
bruto = (
    spark.read.format("csv")
    .option("sep", ";")
    .option("header", True)
    .option("skipRows", 1)      # descarta "Atualizado em: ..."
    .option("inferSchema", False)   # bronze não tipa: tudo string
    .option("quote", "\0")          # desabilita quote
    .option("escape", "\0")         # desabilita escape
    .option("encoding", "UTF-8")
    .option("mode", "PERMISSIVE")   # bronze não descarta linha nenhuma
    .load(CAMINHO)
)

print("Colunas lidas do arquivo: ")
for c in bruto.columns:
    print(f"{c!r}")


# Nomes de coluna: O Delta não aceita espaço
Icao Empresa Aérea é um nome de coluna válido em CSV e **inválido** em Delta -> espaço está proibido( ,;{}()\n\t=).

Então normalizamos o **nome**. Repare que isso não fere a regra da bronze: o que a bronze preserva é o **valor** e a **granularidade**, não a grafia do cabeçalho. Nenhuma coluna é somada, removida, filtrada ou convertida.

O mapa fica explícito no código - nada de `regexo_replace` mágico, para que a correspondência com o arquivo original seja auditável.

In [0]:
RENOMEAR = {
    '"ICAO Empresa Aérea"': "icao_empresa_aerea",
    '"Número Voo"': "numero_voo",
    '"Código Autorização (DI)"': "codigo_autorizacao_di",
    '"Código Tipo Linha"': "codigo_tipo_linha",
    '"ICAO Aeródromo Origem"': "icao_aerodromo_origem",
    '"ICAO Aeródromo Destino"': "icao_aerodromo_destino",
    '"Partida Prevista"': "partida_prevista",
    '"Partida Real"': "partida_real",
    '"Chegada Prevista"': "chegada_prevista",
    '"Chegada Real"': "chegada_real",
    '"Situação Voo"': "situacao_voo",
    '"Código Justificativa"': "codigo_justificativa",
}

FALTANDO = [c for c in RENOMEAR if c not in bruto.columns]
assert not FALTANDO, f"Coluna esperada nao encontrada no CSV: {FALTANDO}"

renomeado = bruto.select(*[F.col(f"`{origem}`").cast("string").alias(novo) for origem, novo in RENOMEAR.items()])

print("Colunas renomeadas: ")
for c in renomeado.columns:
    print(f"{c!r}")

## DBTITLE 1,Verificacao
assert len(bruto.columns) == len(RENOMEAR), "Numero de colunas lidas do arquivo diferente do esperado"
assert len(renomeado.columns) == len(bruto.columns), "Numero de colunas renomeadas diferente do esperado"
assert set(RENOMEAR.values()) == set(renomeado.columns), "Colunas renomeadas diferente do esperado"
assert set(FALTANDO) == set(), "Colunas faltando no CSV"
assert set(RENOMEAR.keys()) == set(bruto.columns), "Colunas renomeadas diferente do esperado"
### DBTITLE 1,Escrita
renomeado.write.mode("overwrite").format("delta").saveAsTable(TABELA)
    
